# META-CXR Stage 1 — private 2×T4 DDP session

One notebook session trains a configurable number of complete epochs, defaults to one, and reserves 90 minutes for private checkpoint publication. The held-out test stays disabled here by default; run the sensitivity notebook once after the final validation-selected checkpoint.

In [ ]:
DATASET_SLUG = "hungnguyen1705/mimic-cxr-jpg-dataset"   # attached private dataset slug
CHECKPOINT_GCS_BUCKET = "meta-cxr-checkpoints-phuongnm"   # GCS bucket for checkpoints (holds one experiment at its root)
CFG_PATH = "pretraining/configs/stage1_smoke_2xt4.yaml"   # the run YAML every cell below drives
REPO_COMMIT = "eb2049f4d3d0faa8cc14771c6b609eb8187b2657"   # validation batch-size stability fix
SESSION_INDEX = 4
SESSION_EPOCHS = 1
TOTAL_PLANNED_EPOCHS = 10                 # fixed LR-schedule horizon across every resume session
SEED = 42
BATCH_PER_GPU = 2   # halved for fp32 activations (run.amp: false)
ACCUMULATION = 32  # doubled so the effective batch stays 2*2*32 = 128
NUM_WORKERS = 4
SESSION_HOURS = 12.0
UPLOAD_RESERVE_MINUTES = 90.0
FINALIZE_TEST = False                     # keep False when notebook 02 will perform final test sensitivity

In [ ]:
import os, pathlib, subprocess, sys
if not DATASET_SLUG or not CHECKPOINT_GCS_BUCKET:
    raise ValueError("DATASET_SLUG and CHECKPOINT_GCS_BUCKET are required")
if len(REPO_COMMIT) != 40 or any(c not in '0123456789abcdef' for c in REPO_COMMIT.lower()):
    raise ValueError("REPO_COMMIT must be an exact 40-character SHA")
repo_dir = pathlib.Path('/kaggle/working/META-CXR-SMOKETEST')
if not repo_dir.exists():
    subprocess.run(['git', 'clone', 'https://github.com/minhphuong150505/META-CXR-SMOKETEST.git', str(repo_dir)], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'fetch', '--depth=1', 'origin', REPO_COMMIT], check=True)
subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', REPO_COMMIT], check=True)
actual = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if actual != REPO_COMMIT:
    raise RuntimeError('Exact commit checkout failed')
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
os.environ['PYTHONPATH'] = str(repo_dir) + os.pathsep + os.environ.get('PYTHONPATH', '')


In [ ]:
# PyTorch 2.6 changed torch.load's default. Resume checkpoints intentionally contain optimizer and per-rank RNG state.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'


In [ ]:
# Load required Kaggle secrets outside /kaggle/working, whose contents become notebook output.
from smoke.runtime import load_kaggle_secrets
load_kaggle_secrets(
    ('GCS_SERVICE_ACCOUNT', 'WANDB_API_KEY'),
    '/tmp/.meta-cxr-secrets',
)
print('Loaded required secrets into OS environment (values hidden).')


In [ ]:
import json
from smoke.runtime import environment_fingerprint, assert_two_t4
before = environment_fingerprint()
print(json.dumps(before, indent=2, sort_keys=True))
assert_two_t4(before)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', 'requirements-kaggle.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-deps', 'hi-ml-multimodal==0.2.1'], check=True)
after = environment_fingerprint()
assert_two_t4(after)
print(json.dumps(after, indent=2, sort_keys=True))


In [ ]:
from smoke.runtime import compatibility_matrix
compatibility = compatibility_matrix(before, after)
print(json.dumps(compatibility, indent=2, sort_keys=True))


In [ ]:
from smoke.runtime import discover_dataset, load_dataset_manifest, write_runtime_env_config
dataset_root = discover_dataset(DATASET_SLUG)
dataset_manifest, manifest_path, dataset_hash = load_dataset_manifest(dataset_root)
if dataset_manifest.get('status') != 'qa_passed':
    raise RuntimeError('Dataset manifest is not QA-passed')
write_runtime_env_config(dataset_root, '/kaggle/working/meta-cxr-output')
subprocess.run([sys.executable, 'scripts/kaggle_data_preflight.py', '--dataset-root', str(dataset_root), '--seed', str(SEED)], check=True)
print({'dataset_manifest_sha256': dataset_hash, 'actual_bytes': dataset_manifest.get('actual_bytes'), 'counts': dataset_manifest.get('counts')})


In [ ]:
import hashlib, json, math, shutil, torch
if TOTAL_PLANNED_EPOCHS < SESSION_EPOCHS:
    raise ValueError('TOTAL_PLANNED_EPOCHS must cover the requested session')
# config_fingerprint is the *scientific* identity of the run: dataset + encoders
# + resolution + micro-batching + seed + schedule horizon + label policy. The
# git commit (source_commit) is deliberately NOT hashed in here, so a
# plumbing/code-only commit that leaves this config unchanged still resumes.
identity_payload = {
    'dataset_manifest_sha256': dataset_hash,
    'encoders': ['biovil', 'pubmedclip', 'swin'], 'multi_view': True,
    'image_size': 448, 'batch_per_gpu': BATCH_PER_GPU, 'world_size': 2,
    'accumulation': ACCUMULATION, 'seed': SEED, 'scheduler_max_epoch': TOTAL_PLANNED_EPOCHS,
    # Hashed for historical continuity only. The LIVE selection metric is
    # run.selection_metric in the YAML; it decides which epoch
    # checkpoint_best.pth keeps, not the training trajectory, so changing
    # it must NOT invalidate a resume.
    'selection_metric': 'f1_positive_macro_defined_only',
    'uncertain_policy': 'ignore_uncertain',
}
config_fingerprint = hashlib.sha256(json.dumps(identity_payload, sort_keys=True).encode()).hexdigest()
# Stable per-experiment slug (dataset+seed, NOT per session) so every session
# resumes the same W&B run. Checkpoints live at the GCS bucket root, so this
# bucket holds exactly one experiment.
run_name = f"meta-cxr-e123-{dataset_hash[:8]}-seed{SEED}"
wandb_run_id = run_name
output_base = pathlib.Path('/kaggle/working/meta-cxr-output')
run_dir = output_base / run_name
start_epoch = 0
# Auto-resume: pull only checkpoint_last.pth from the GCS bucket root if a prior session ran.
from smoke.checkpoints import download_last_checkpoint
resume_path = download_last_checkpoint(CHECKPOINT_GCS_BUCKET, run_dir)
if resume_path is not None:
    prior = torch.load(resume_path, map_location='cpu')
    prior_id = prior.get('identity', {})
    if prior_id.get('dataset_manifest_sha256') != dataset_hash:
        raise RuntimeError('Prior checkpoint was trained on a different dataset; refusing to resume')
    if prior_id.get('config_fingerprint') != config_fingerprint:
        raise RuntimeError('Prior checkpoint has a different training config (encoders/image_size/batch/accumulation/seed/scheduler horizon/label policy); refusing to resume. Use a fresh GCS bucket for a new configuration.')
    # Same version floor the runner enforces, applied HERE so a stranded
    # checkpoint costs five seconds instead of a full preflight. nb01 keeps its
    # own copy of the resume decision, so without this it happily computes a
    # start_epoch that pretraining.train will refuse minutes later.
    from smoke.resume import assert_resumable_version
    try:
        assert_resumable_version(prior, str(resume_path))
    except RuntimeError as exc:
        raise RuntimeError(
            str(exc)
            + f"\n\nThis file is at the bucket root and will be re-downloaded every "
            f"session until it is moved. To start the fresh run:\n"
            f"  gsutil mv gs://{CHECKPOINT_GCS_BUCKET}/checkpoint_last.pth "
            f"gs://{CHECKPOINT_GCS_BUCKET}/retired-v2/checkpoint_last.pth\n"
            f"  rm -rf {run_dir}\n"
            "then re-run this cell."
        ) from None
    prior_commit = prior.get('source_commit')
    if prior_commit and prior_commit != REPO_COMMIT:
        print('Note: resuming a checkpoint produced by commit', prior_commit, '- dataset + config match, so resume is safe across this code change')
    start_epoch = int(prior['epoch']) + 1
    print('Resuming from GCS checkpoint at epoch', int(prior['epoch']), '-> start_epoch', start_epoch)
else:
    print('No checkpoint_last.pth at the GCS bucket root - starting from epoch 0')
max_epoch = start_epoch + SESSION_EPOCHS
if max_epoch > TOTAL_PLANNED_EPOCHS:
    raise ValueError('Session would exceed TOTAL_PLANNED_EPOCHS')
train_studies = int(dataset_manifest['counts']['split_studies']['train'])
microbatches_per_rank = math.ceil(train_studies / 2 / BATCH_PER_GPU)
optimizer_steps = math.ceil(microbatches_per_rank / ACCUMULATION)
warmup_steps = max(1, math.ceil(optimizer_steps * 0.10))
print({'run_name': run_name, 'wandb_run_id': wandb_run_id, 'start_epoch': start_epoch, 'max_epoch': max_epoch, 'scheduler_max_epoch': TOTAL_PLANNED_EPOCHS, 'optimizer_steps_per_epoch': optimizer_steps, 'effective_batch': BATCH_PER_GPU * 2 * ACCUMULATION, 'warmup_steps': warmup_steps})


In [ ]:
# Repair a collapsed ITC temperature before resuming.
#
# `temp` is the learnable contrastive temperature. ITC always pushes it down to
# sharpen the softmax, and in the e123 run it fell to 5.19e-4 -- straight
# through the 1e-3 clamp floor that was in force at the time. Because ITC uses
# L2-normalized features its loss is bounded by 2/T + ln(N), which is ~36 at
# T=0.07 but ~2007 at T=1e-3, and epoch 2 logged bursts up to 171.9. With
# max_grad_norm=1.0 those bursts scale down every *other* objective's gradient
# too, so ITM/LM/classification learn almost nothing while one is happening.
#
# Commit bc9df94 raised the floor to 0.01, which caps the damage -- but it does
# not repair it. `clamp` has zero gradient outside its range, so a parameter
# sitting below the floor is frozen: epoch 3 ran the whole 657 optimizer steps
# with the new floor and `temp` came out bit-identical at 5.192368e-04, its
# Adam exp_avg decayed to 4.2e-43. Left alone it stays pinned forever.
#
# So: put `temp` back at its 0.07 init, inside the clamp range where it can
# learn again, and drop its Adam moments. exp_avg_sq was 0.21; carrying that
# denominator forward would damp every future `temp` gradient to nearly
# nothing, which is the same freeze by another route.
#
# The guard is `temp < floor`. That is exactly the stuck-below-the-floor
# condition and nothing else, so this is a no-op on a healthy checkpoint and on
# a re-run of this cell. Patching happens on the local copy the runner resumes
# from; the GCS object is untouched until training uploads epoch 5.
ITC_TEMP_FLOOR = 0.01   # must match the clamp min in blip2_qformer.Blip2Qformer.forward
ITC_TEMP_INIT = 0.07    # must match self.temp = nn.Parameter(0.07 * torch.ones([]))

if resume_path is None:
    print('No resume checkpoint - nothing to repair')
else:
    ckpt = torch.load(resume_path, map_location='cpu')
    temp_keys = [k for k in ckpt['model'] if k.split('.')[-1] == 'temp']
    if len(temp_keys) != 1:
        raise RuntimeError(f'Expected exactly one temp parameter, found {temp_keys}')
    temp_key = temp_keys[0]
    temp_value = float(ckpt['model'][temp_key].item())
    if temp_value >= ITC_TEMP_FLOOR:
        print(f'{temp_key}={temp_value:.6e} is inside the clamp range - no repair needed')
    else:
        ckpt['model'][temp_key] = torch.full_like(ckpt['model'][temp_key], ITC_TEMP_INIT)
        # temp is the only 0-dim parameter in the model, so it owns the only
        # 0-dim entry in the optimizer state.
        opt_state = ckpt['optimizer']['state']
        zero_dim = [i for i, s in opt_state.items()
                    if torch.is_tensor(s.get('exp_avg')) and s['exp_avg'].ndim == 0]
        if len(zero_dim) != 1:
            raise RuntimeError(f'Expected exactly one 0-dim optimizer state, found {zero_dim}')
        dropped = opt_state.pop(zero_dim[0])
        torch.save(ckpt, resume_path)
        print({'repaired': temp_key,
               'temp_before': temp_value, 'temp_after': ITC_TEMP_INIT,
               'dropped_adam_state_index': zero_dim[0],
               'dropped_exp_avg_sq': float(dropped['exp_avg_sq'].item()),
               'resumed_epoch': int(ckpt['epoch'])})
    del ckpt


In [ ]:
# Two-rank construction + forward/backward/all-reduce/checkpoint preflight (2 optimizer steps).
preflight_name = f'preflight-{dataset_hash[:8]}-seed{SEED}'
preflight_dir = pathlib.Path('/kaggle/working/meta-cxr-preflight') / preflight_name
preflight_fingerprint = hashlib.sha256((config_fingerprint + ':preflight').encode()).hexdigest()
preflight_cmd = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', '-m', 'pretraining.train', '--cfg-path', CFG_PATH, '--options',
    f'run.run_name={preflight_name}', 'run.output_dir=/kaggle/working/meta-cxr-preflight', f'run.source_commit={REPO_COMMIT}', f'run.dataset_manifest_sha256={dataset_hash}', f'run.config_fingerprint={preflight_fingerprint}',
    f'run.batch_size_train={BATCH_PER_GPU}', f'run.accum_grad_iters={ACCUMULATION}', f'run.num_workers={NUM_WORKERS}', 'run.warmup_steps=1', 'run.max_epoch=1', f'run.truncate_train={BATCH_PER_GPU * 2 * ACCUMULATION * 2}', 'run.truncate_val=8', 'run.test_splits=[]', 'run.finalize_test=false', 'run.save_predictions=false', 'run.run_role=preflight']
# PYTHONUNBUFFERED so the child process streams stdout live; without it the
# printed step count block-buffers and lags far behind real training progress.
unbuffered_env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
if not (preflight_dir / 'checkpoint_last.pth').is_file():
    if preflight_dir.exists():
        shutil.rmtree(preflight_dir)   # discard partial output from a prior failed preflight
    from smoke.proc import stream_and_capture
    # Log file lives beside preflight_dir, not inside it -- stream_and_capture
    # creates the file before the subprocess starts, and a file inside
    # preflight_dir would make setup_output_dir's non-empty-run-dir guard
    # reject the run before training even begins.
    stream_and_capture(preflight_cmd, preflight_dir.parent / f'{preflight_name}_preflight_console.log', env=unbuffered_env)
log_lines = (preflight_dir / 'log.txt').read_text().splitlines()
train_logs = [json.loads(line) for line in log_lines if line.startswith('{"train_') and 'train_epoch_wall_seconds' in line]
if not train_logs:
    raise RuntimeError('DDP preflight did not emit timing evidence')
probe = train_logs[-1]
if not math.isfinite(float(probe['train_loss'])):
    raise RuntimeError('DDP preflight loss is not finite')
# A finite loss is NOT evidence that training works. Run e123 finished two full
# epochs with a finite loss, zero applied optimizer steps and an AMP scale that
# had collapsed to 0: every step was skipped by the overflow guard, so the run
# saved checkpoints that contained the initial weights. Gate on the counters
# that say an update was actually applied.
steps_taken = int(probe.get('train_optimizer_steps_taken', -1))
overflows = int(probe.get('train_amp_overflow_steps', -1))
collapses = int(probe.get('train_amp_collapse_resets', -1))
if steps_taken < 1:
    raise RuntimeError(
        f'DDP preflight applied {steps_taken} optimizer step(s): the weights never '
        'moved. A full run started from here would train nothing. Investigate fp16 '
        'stability (loss magnitude, encoder output range) before launching.')
if overflows != 0 or collapses != 0:
    raise RuntimeError(
        f'DDP preflight hit {overflows} AMP overflow step(s) and {collapses} scale '
        'collapse reset(s). On a 2-step preflight that means gradients are '
        'non-finite from the very first window, not an occasional fp16 spike.')
print(probe)

In [ ]:
from smoke.runtime import assert_session_eta
eta = assert_session_eta(int(probe['train_optimizer_steps']), float(probe['train_epoch_wall_seconds']), optimizer_steps, SESSION_HOURS, UPLOAD_RESERVE_MINUTES)
print(eta)


In [ ]:
full_cmd = [sys.executable, '-m', 'torch.distributed.run', '--standalone', '--nproc_per_node=2', '-m', 'pretraining.train', '--cfg-path', CFG_PATH, '--options',
    f'run.run_name={run_name}', f'run.source_commit={REPO_COMMIT}', f'run.dataset_manifest_sha256={dataset_hash}', f'run.config_fingerprint={config_fingerprint}',
    f'run.batch_size_train={BATCH_PER_GPU}', f'run.accum_grad_iters={ACCUMULATION}', f'run.num_workers={NUM_WORKERS}', f'run.warmup_steps={warmup_steps}', f'run.max_epoch={max_epoch}', f'run.scheduler_max_epoch={TOTAL_PLANNED_EPOCHS}', f'run.seed={SEED}', f'run.finalize_test={str(FINALIZE_TEST).lower()}',
    f'run.wandb_run_id={wandb_run_id}', 'run.run_role=train']
if resume_path is not None:
    full_cmd.append(f'run.resume_ckpt_path={resume_path}')
completed = False
if (run_dir / 'checkpoint_last.pth').is_file():
    current = torch.load(run_dir / 'checkpoint_last.pth', map_location='cpu')
    completed = int(current.get('epoch', -1)) >= max_epoch - 1 and current.get('identity', {}).get('config_fingerprint') == config_fingerprint
if not completed:
    # PYTHONUNBUFFERED so the training log streams live in the notebook instead
    # of block-buffering (which made the step count appear stuck far behind).
    from smoke.proc import stream_and_capture
    _env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    # Task F: opt-in NCCL flight recorder + per-rank markers. Set the Kaggle
    # env var DISTRIBUTED_DEBUG=1 to capture a collective trace on a hang.
    if os.environ.get('DISTRIBUTED_DEBUG', '').lower() in ('1', 'true', 'yes'):
        _env.update({'META_DDP_DEBUG': '1', 'NCCL_DEBUG': 'INFO',
                     'TORCH_NCCL_TRACE_BUFFER_SIZE': '1048576',
                     'TORCH_NCCL_DUMP_ON_TIMEOUT': '1'})
    # Tee live output to the notebook AND to a console log file beside run_dir
    # (not inside it) -- stream_and_capture creates the file before training
    # starts, and a file inside run_dir would make setup_output_dir's
    # non-empty-run-dir guard reject a fresh (non-resumed) run before it begins.
    # A lost Kaggle session still leaves the final traceback on disk.
    stream_and_capture(full_cmd, run_dir.parent / f'{run_name}_full_train_console.log', env=_env)
last = torch.load(run_dir / 'checkpoint_last.pth', map_location='cpu')
if int(last['epoch']) != max_epoch - 1:
    raise RuntimeError('Session did not finish the requested complete epoch count')
print({'completed_epoch': int(last['epoch']), 'run_dir': str(run_dir), 'peak_vram_each_rank': 'recorded in Kaggle nvidia-smi/runtime logs'})

In [ ]:
import yaml
run_cfg_yaml = yaml.safe_load(pathlib.Path(CFG_PATH).read_text())
run_manifest = {
    'source_commit': REPO_COMMIT,
    'dataset_manifest_sha256': dataset_hash,
    'config_fingerprint': config_fingerprint,
    'session_index': SESSION_INDEX,
    'completed_epoch': int(last['epoch']),
    'training': {
        'encoders': ['biovil', 'pubmedclip', 'swin'], 'multi_view': True,
        'image_size': 448, 'batch_per_gpu': BATCH_PER_GPU, 'world_size': 2,
        'accumulation': ACCUMULATION, 'effective_batch': BATCH_PER_GPU * 2 * ACCUMULATION,
        'optimizer_steps_per_epoch': optimizer_steps, 'scheduler_max_epoch': TOTAL_PLANNED_EPOCHS,
        'warmup_steps': warmup_steps, 'amp': 'fp16', 'seed': SEED, 'num_workers': NUM_WORKERS,
        # Read from the YAML the runner is actually driven by. Hardcoding it here
        # let the manifest claim 'loss_cls' while run.selection_metric said
        # 'loss', so the recorded provenance named the wrong objective for
        # checkpoint_best.pth.
        'selection_metric': run_cfg_yaml['run']['selection_metric'],
        'selection_mode': run_cfg_yaml['run']['selection_mode'],
        'uncertain_policy': 'ignore_uncertain',
    },
    'eta_probe': eta,
    'environment': after,
    'held_out_test_finalized': FINALIZE_TEST,
}
(run_dir / 'run_manifest.json').write_text(json.dumps(run_manifest, indent=2, sort_keys=True))


In [ ]:
from smoke.checkpoints import upload_checkpoint_gcs
uri = upload_checkpoint_gcs(CHECKPOINT_GCS_BUCKET, run_dir)
print('Uploaded checkpoint_last.pth + checkpoint_best.pth (if any) + log.txt to', uri)
print('Next session auto-resumes this experiment from checkpoint_last.pth')
